# TCGA Cancer Embedding Visualization Notebook

## Pull Embeddings

In [ ]:
# TODO replace this entire cell with getting the embeddings from the API
# !wget "https://uchicago.box.com/shared/static/k8z0kip2pej2v62pwgymdm45gt7dq0yw.h5" -O "data/hist.h5"
# !wget "https://uchicago.box.com/shared/static/hr82b5c9g3h4y8c7avrnbgvhgdcoetld.h5" -O "data/expr.h5"
# !wget "https://uchicago.box.com/shared/static/liwt3vlvdpmbfsa21wqboshh9nv6enm2.h5" -O "data/summ.h5"

import h5py
from tqdm import tqdm

expr_file = "with-all-data/data/expr.h5" # BulkRNABert
hist_file = "with-all-data/data/hist.h5" # UNI2
text_file = "with-all-data/data/summ.h5" # BioMistral - Summarized

expr_embs = dict()
hist_embs = dict()
text_embs = dict()

for fpath, embs, do_upper in tqdm([
    (expr_file, expr_embs, False),
    (hist_file, hist_embs, False),
    (text_file, text_embs, True),
]):
    with h5py.File(fpath, "r") as h5:
        for case_id in h5.keys():
            for sample_fname in h5[case_id].keys():
                emb_key = sample_fname
                if do_upper:
                    emb_key = emb_key.upper()
                embs[emb_key] = h5[case_id][sample_fname][:]

## Pull and Join Metadata

In [ ]:
import requests
import numpy as np
import pandas as pd
from io import StringIO

tcga_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}]}
hist_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}, {"op": "in", "content": {"field": "files.experimental_strategy", "value": ["Diagnostic Slide"]}}, {"op": "in", "content": {"field": "files.data_format", "value": ["svs"]}}]}
expr_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}, {"op": "in", "content": {"field": "files.experimental_strategy", "value": ["RNA-Seq"]}}, {"op": "in", "content": {"field": "files.data_type", "value": ["Gene Expression Quantification"]}}, {"op": "in", "content": {"field": "files.data_format", "value": ["tsv"]}}]}
text_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}, {"op": "in", "content": {"field": "files.data_type", "value": ["Pathology Report"]}}, {"op": "in", "content": {"field": "files.data_format", "value": ["pdf"]}}]}

# case metadata
response = requests.post(
    "https://api.gdc.cancer.gov/cases",
    json={
        "filters": tcga_filter,
        "fields": ",".join(["project.project_id", "submitter_id", "diagnoses.age_at_diagnosis", "diagnoses.diagnosis_is_primary_disease", "demographic.sex_at_birth", "demographic.race", "demographic.ethnicity"]),
        "format": "JSON",
        "size": str(100_000),
    },
)

hits = []
for hit in response.json()["data"]["hits"]:
    proj = hit.pop("project", {})
    demo = hit.pop("demographic", {})
    dxs = hit.pop("diagnoses", [])
    for dx in dxs:
        if dx.get("diagnosis_is_primary_disease", False) and dx["age_at_diagnosis"] is not None and not np.isnan(dx["age_at_diagnosis"]):
            hit["age_at_diagnosis"] = dx["age_at_diagnosis"]
            break
    if "age_at_diagnosis" in hit:
        hits.append(hit | proj | demo)
metadata = pd.DataFrame(hits).drop(columns=["id"])
assert metadata["age_at_diagnosis"].notna().all()
metadata["age_in_years"] = metadata["age_at_diagnosis"] / 365.24 # convert age to years
metadata["age_binned"] = pd.cut(metadata["age_in_years"], bins=[0, 20, 40, 60, 80, 100]) # convert age to 20-year bins

# survival data
response = requests.post(
    "https://api.gdc.cancer.gov/analysis/survival",
    json={"filters": tcga_filter},
)
rows = response.json()["results"][0]["donors"]
survival = pd.DataFrame(rows).drop(columns=["id", "project_id"])

# get mapping of file ID --> case ID
def get_file_mapping(cohort_filter):
    response = requests.post(
        "https://api.gdc.cancer.gov/files",
        json={
            "filters": cohort_filter,
            "fields": ",".join(["file_name", "cases.project.project_id", "cases.submitter_id"]),
            "format": "TSV",
            "size": str(100_000),
        },
    )
    df = pd.read_csv(StringIO(response.text), sep="\t")
    df = df.rename(columns={"cases.0.project.project_id": "project_id", "cases.0.submitter_id": "submitter_id"}) # these files all map to a single case
    df["file_name"] = df["file_name"].str.replace(".svs", "").str.replace(".rna_seq.augmented_star_gene_counts.tsv", "").str.replace(".PDF", "")
    assert df["file_name"].is_unique
    return df.set_index("submitter_id")["file_name"]

hist_mapping = get_file_mapping(hist_filter)
expr_mapping = get_file_mapping(expr_filter)
text_mapping = get_file_mapping(text_filter)

case_ids = (
    # case IDs that have all 3 embedding modalities
    set(expr_mapping[expr_mapping.isin(expr_embs)].index) &
    set(hist_mapping[hist_mapping.isin(hist_embs)].index) &
    set(text_mapping[text_mapping.isin(text_embs)].index)
)

assert metadata["submitter_id"].is_unique and survival["submitter_id"].is_unique
df = metadata.merge(survival, on="submitter_id")
df = df[df["submitter_id"].isin(case_ids)]
df = df.sort_values(["project_id", "submitter_id"])
df = df.set_index("submitter_id").copy()

# align ordering of embeddings to case order in metadata
expr_X = []
hist_X = []
text_X = []
for case_id in tqdm(df.index):
    for embs, mapping, X in [
        (expr_embs, expr_mapping, expr_X),
        (hist_embs, hist_mapping, hist_X),
        (text_embs, text_mapping, text_X),
    ]:
        fnames = mapping.loc[case_id]
        if isinstance(fnames, str): # single file, guaranteed to be in embs
            emb = embs[fnames]
        else: # multiple files, at least one will be in embs
            emb = np.mean([embs[f] for f in fnames if f in embs], axis=0)
        X.append(emb)
expr_X = np.asarray(expr_X)
hist_X = np.asarray(hist_X)
text_X = np.asarray(text_X)

## Prepare Data for Visualization

In [ ]:
df["race"].replace({
    "white": "White",
    "asian": "Asian",
    "not reported": "Not Reported",
    "black or african american": "Black",
    "american indian or alaska native": "AIAN",
    "native hawaiian or other pacific islander": "NHPI",
}, inplace=True)

df["ethnicity"].replace({
    "not hispanic or latino": "Not Hispanic/Latino",
    "not reported": "Not Reported",
    "hispanic or latino": "Hispanic/Latino",
}, inplace=True)

df["vital_status"] = np.where(df["censored"], "Alive", "Dead").tolist()

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pipe = Pipeline(
    [
        ("scale_raw", StandardScaler()),
        ("pca_reduce", PCA(32, random_state=42)), # reduce dimensionality so UMAP runs faster
        ("scale_pca", StandardScaler()), # normalize PCA output
    ],
)

modality_map = {
    "expr": clone(pipe).fit_transform(expr_X),
    "hist": clone(pipe).fit_transform(hist_X),
    "text": clone(pipe).fit_transform(text_X),
}

output_types = {
    "project_id": "cat",
    "age_binned": "cat",
    "sex_at_birth": "cat",
    "race": "cat",
    "ethnicity": "cat",
    "vital_status": "cat",
    "age_in_years": "cont",
    "time": "cont",
}

cont_intervals = {
    "age_in_years": 15,
    "time": 2000,
}

long_legends = {
    "project_id",
}

inputs = list(modality_map.keys())
cat_outputs = [k for k, v in output_types.items() if v == "cat"]
cont_outputs = [k for k, v in output_types.items() if v == "cont"]

## Visualization Widget

In [ ]:
from umap import UMAP
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import seaborn as sns
from ipywidgets import Checkbox, HBox, VBox, RadioButtons, Output, Label, Layout, HTML, interactive_output

%matplotlib widget

controls = {
    name: Checkbox(value=i==0, description=name, indent=False)
    for i, name in enumerate(inputs)
}

controls["color_selector"] = RadioButtons(
    options=cat_outputs + cont_outputs,
    value="project_id",
    description="Color by:",
)

controls["n_dims"] = RadioButtons(
    options=[2, 3],
    value=3,
    description="Number of Dimensions",
)

output_plot = Output()

spinner_html = """
<div style="display: flex; justify-content: center; align-items: center; width: 100%; height: 500px;">
    <div class="loader"></div>
</div>

<style>
.loader {
  border: 6px solid #f3f3f3;
  border-top: 6px solid #3498db;
  border-radius: 50%;
  width: 30px;
  height: 30px;
  animation: spin 1s linear infinite;
}

@keyframes spin {
  0% { transform: rotate(0deg); }
  100% { transform: rotate(360deg); }
}
</style>
"""

cached_modalities = dict()

def update(color_selector, n_dims, **modality_selections):
    with output_plot:
        output_plot.clear_output(wait=False)
        display(HTML(spinner_html))
        output_plot.clear_output(wait=True)
        plt.close("all")

        modalities = [name for name, checked in modality_selections.items() if checked]
        modalities = sorted(modalities)
        modalities_str = ", ".join(modalities)

        if modalities_str == "":
            print("Please select at least 1 modality")
            return

        modalities_str += f"_{n_dims}d"

        if modalities_str not in cached_modalities:
            X = np.concatenate([modality_map[modality] for modality in modalities], axis=1)
            cached_modalities[modalities_str] = UMAP(n_components=n_dims, n_neighbors=30, n_jobs=4).fit_transform(X)
        reduced = cached_modalities[modalities_str]

        output_type = output_types[color_selector]
        if output_type == "cont":
            cmap = sns.color_palette("viridis", as_cmap=True)
            vmin = df[color_selector].min()
            vmax = df[color_selector].max()
            norm = Normalize(vmin=vmin, vmax=vmax)
            colors = cmap(norm(df[color_selector]))
            interval = cont_intervals[color_selector]
            lowest = ((vmin + interval - 1) // interval) * interval
            count = int((vmax - lowest) // interval) + 1
            legend_elements = []
            for i in range(count):
                val = int(i * interval + lowest)
                legend_elements.append(Line2D([0], [0], marker="o", ls="none", color=cmap(norm(val)), label=val))
        elif output_type == "cat":
            temp = pd.Categorical(df[color_selector])
            codes = temp.codes
            categories = dict(enumerate(temp.categories))
            n_unique = codes.max() + 1
            if n_unique > 10:
                cmap = sns.color_palette("husl", n_unique)
            else:
                cmap = sns.color_palette("tab10")
            colors = np.asarray([cmap[i] for i in codes])
            legend_elements = []
            for i in range(n_unique):
                legend_elements.append(Line2D([0], [0], marker="o", ls="none", color=cmap[i], label=categories[i]))
        else:
            raise ValueError(f"Unknown output_type: {output_type}")

        fig = plt.figure(figsize=(9, 7), num=" ")
        gs = GridSpec(1, 9, figure=fig)
        ax = fig.add_subplot(gs[0, 0:7], projection=None if n_dims == 2 else "3d")
        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")

        args = [reduced[:, 0], reduced[:, 1]]
        if n_dims > 2:
            args.append(reduced[:, 2])
            ax.set_zlabel("UMAP-3")

        ax.scatter(*args, c=colors, s=1)

        ax2 = fig.add_subplot(gs[0, 7:9])
        ax2.set_xticks([])
        ax2.set_yticks([])
        ax2.axis("off")
        title = " ".join(color_selector.split("_")).title().replace("Fu", "FU").replace("Id", "ID")
        ax2.legend(
            handles=legend_elements,
            loc="center",
            frameon=True,
            title=title,
            fontsize=8 if color_selector in long_legends else 10,
        )

        fig.tight_layout()
        plt.show()

interactive = interactive_output(update, controls)
checkbox_column = VBox([Label("Embedding Modality:")] + list(controls.values()), layout=Layout(width="150px"))
ui = HBox([checkbox_column, VBox([output_plot], layout=Layout(width="1000px"))])
display(ui, interactive)